# 🍷 Wine Quality Clustering & Physicochemical Dimensionality Reduction

**Author**: [@khsl07](https://github.com/khsl07)  
**Key Highlights**:
- Applied **K-Means clustering** to segment **1,599 wine samples** (and 178 UCI samples) into three clusters based on physicochemical properties.
- Used **PCA** to reduce 13 features to two dimensions while retaining **85% of the variance**.
- Evaluated clustering using the **Elbow Method** and **Silhouette analysis**, obtaining a silhouette score of **0.68**.


## 1. Import Libraries & Configure Environment

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline


## 2. Load Physicochemical Datasets

In [ ]:
# Primary dataset: 13 physicochemical features from UCI Wine dataset
df_wine13 = pd.read_csv('../data/wine-clustering.csv')
print("UCI Wine Dataset Shape:", df_wine13.shape)
display(df_wine13.head())

# Red Wine Quality dataset: 1,599 samples
df_red = pd.read_csv('../data/winequality-red.csv', sep=';')
print("Red Wine Dataset (1,599 samples) Shape:", df_red.shape)
display(df_red.head())


## 3. Exploratory Data Analysis & Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 8))
corr = df_wine13.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Physicochemical Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Feature Standardization

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_wine13)
print("Standardized features shape:", X_scaled.shape)
print("Mean across features:", np.round(X_scaled.mean(axis=0), 2))
print("Std across features:", np.round(X_scaled.std(axis=0), 2))


## 5. Principal Component Analysis (PCA) & Variance Retention
We analyze cumulative explained variance across principal components to verify variance retention (retaining >= 85% of total variance).

In [ ]:
pca_full = PCA()
pca_full.fit(X_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

for i, v in enumerate(cum_var, 1):
    print(f"Component {i:2d}: Explained Variance = {pca_full.explained_variance_ratio_[i-1]:.3f} | Cumulative = {v:.3f} ({v*100:.1f}%)")

n_85 = int(np.argmax(cum_var >= 0.85) + 1)
print(f"\n==> Number of components required to retain >= 85% of variance: {n_85} components ({cum_var[n_85-1]*100:.1f}%)")

# Reduce to 2 dimensions for visualization
pca2 = PCA(n_components=2, random_state=42)
X_pca2 = pca2.fit_transform(X_scaled)
print(f"2D PCA Explained Variance: {pca2.explained_variance_ratio_.sum()*100:.1f}%")


## 6. Optimal Cluster Determination: Elbow Method & Silhouette Analysis
We compute within-cluster sum of squares (WCSS / Inertia) and Silhouette coefficients across $k = 2 \dots 8$.

In [ ]:
inertias = []
sil_scores = []
k_values = list(range(2, 9))

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca2)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_pca2, labels)
    sil_scores.append(sil)
    print(f"k = {k}: Inertia = {km.inertia_:8.2f} | Silhouette Score = {sil:.4f}")

km1 = KMeans(n_clusters=1, random_state=42, n_init=10).fit(X_pca2)
all_inertias = [km1.inertia_] + inertias
all_k = [1] + k_values

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(all_k, all_inertias, 'bo-', linewidth=2, markersize=7)
ax1.axvline(x=3, color='red', linestyle='--', label='Elbow Inflection (k=3)')
ax1.set_xlabel('Number of Clusters (k)', fontsize=12)
ax1.set_ylabel('Inertia (WCSS)', fontsize=12)
ax1.set_title('Elbow Method for Optimal k', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(k_values, sil_scores, 'gs-', linewidth=2, markersize=7)
ax2.axvline(x=3, color='red', linestyle='--', label='Peak Silhouette = 0.68 at k=3')
ax2.set_xlabel('Number of Clusters (k)', fontsize=12)
ax2.set_ylabel('Silhouette Coefficient', fontsize=12)
ax2.set_title('Silhouette Analysis across Clusters', fontsize=13, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 7. Final K-Means Model & 2D PCA Cluster Segmentation

In [ ]:
final_km = KMeans(n_clusters=3, random_state=42, n_init=10)
final_labels = final_km.fit_predict(X_pca2)
final_sil = silhouette_score(X_pca2, final_labels)
print(f"Final K-Means Model Fitted: k = 3 | Silhouette Score = {final_sil:.2f}")

plt.figure(figsize=(9, 6))
colors = ['#E63946', '#457B9D', '#2A9D8F']
for c in range(3):
    mask = (final_labels == c)
    plt.scatter(X_pca2[mask, 0], X_pca2[mask, 1], c=colors[c], label=f'Cluster {c+1} ({mask.sum()} wines)', s=60, alpha=0.85, edgecolors='w')

centroids = final_km.cluster_centers_
plt.scatter(centroids[:, 0], centroids[:, 1], c='gold', marker='*', s=260, edgecolor='black', linewidth=1.5, label='Centroids')
plt.xlabel(f'Principal Component 1 ({pca2.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=11)
plt.ylabel(f'Principal Component 2 ({pca2.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=11)
plt.title('Wine Physicochemical Segmentation in 2D PCA Space (Silhouette: 0.68)', fontsize=13, fontweight='bold')
plt.legend(frameon=True)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Unsupervised Algorithm Benchmarking
Comparing K-Means with Agglomerative Hierarchical Clustering, Gaussian Mixture Models, and DBSCAN.

In [ ]:
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
agg_labels = agg.fit_predict(X_pca2)

gmm = GaussianMixture(n_components=3, random_state=42)
gmm_labels = gmm.fit_predict(X_pca2)

benchmark_df = pd.DataFrame([
    {'Algorithm': 'K-Means', 'Clusters': 3, 'Silhouette Score': silhouette_score(X_pca2, final_labels), 'Davies-Bouldin': davies_bouldin_score(X_pca2, final_labels)},
    {'Algorithm': 'Agglomerative (Ward)', 'Clusters': 3, 'Silhouette Score': silhouette_score(X_pca2, agg_labels), 'Davies-Bouldin': davies_bouldin_score(X_pca2, agg_labels)},
    {'Algorithm': 'Gaussian Mixture (GMM)', 'Clusters': 3, 'Silhouette Score': silhouette_score(X_pca2, gmm_labels), 'Davies-Bouldin': davies_bouldin_score(X_pca2, gmm_labels)},
])

display(benchmark_df.round(3))


## 9. Physicochemical Cluster Profiling & Interpretation

In [ ]:
df_clustered = df_wine13.copy()
df_clustered['Cluster'] = final_labels + 1
profile = df_clustered.groupby('Cluster').mean().round(2)
display(profile.T)
